# MonIA Kaggle Video Studio
GPU worker for **candidate-only** Marion & Lucas video generation.

Rules: gameplay remains narrative authority; generated media never goes live automatically; motion references never provide identity.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors imageio[ffmpeg] huggingface_hub requests pillow

In [ ]:
import os, json, time, hashlib, pathlib, requests, torch
from pathlib import Path

REPO = os.environ.get('MONIA_REPO', 'vartcom38-collab/marion-lucas-game')
BRANCH = os.environ.get('MONIA_BRANCH', 'main')
RAW = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}'
WORK = Path('/kaggle/working/monia-studio')
WORK.mkdir(parents=True, exist_ok=True)
print('MonIA Studio ready · candidate-only')

In [ ]:
def load_json(url):
    r=requests.get(url,timeout=30); r.raise_for_status(); return r.json()

def download(url, target):
    r=requests.get(url,timeout=90); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)

def validate_job(job):
    assert job.get('candidateOnly') is True, 'candidateOnly must be true'
    assert job.get('narrativeAuthority') is False, 'GPU worker cannot have narrative authority'
    assert job.get('source') in {'gameplay','anticipation','review-regeneration'}
    motion=job.get('motion') or {}
    assert motion.get('copyIdentity',False) is False, 'motion reference identity copy forbidden'
    assert job.get('output',{}).get('candidatePath'), 'candidate output path required'
    return job


## Job input
Set `MONIA_JOB_URL` to a raw GitHub JSON job. The notebook processes **one heavy job at a time**.

In [ ]:
JOB_URL=os.environ.get('MONIA_JOB_URL','').strip()
if not JOB_URL:
    raise RuntimeError('Set MONIA_JOB_URL to a queued MonIA Studio job JSON')
job=validate_job(load_json(JOB_URL))
job_id=job['id']
job_dir=WORK/job_id
job_dir.mkdir(parents=True,exist_ok=True)
(job_dir/'job.json').write_text(json.dumps(job,ensure_ascii=False,indent=2),encoding='utf-8')
print('Loaded',job_id,job.get('sceneFamily'))

In [ ]:
# Resolve canonical character references and optional motion/continuity sources.
refs={}
for ch in job.get('characters',[]):
    url=ch['canonRef']
    if url.startswith('/'):
        url=RAW+url
    refs[ch['id']]=download(url,job_dir/f"canon-{ch['id']}.jpg")

continuity=(job.get('continuity') or {}).get('previousFrameUrl')
if continuity:
    if continuity.startswith('/'): continuity=RAW+continuity
    refs['previous_frame']=download(continuity,job_dir/'previous-frame.png')

print('Canonical refs:',refs)

## Generation router
The worker deliberately keeps model loading isolated. `auto` should prefer a model that fits the active Kaggle GPU. Wan/LTX adapters can be upgraded without changing the gameplay contract.

In [ ]:
def gpu_info():
    if not torch.cuda.is_available(): return {'available':False}
    p=torch.cuda.get_device_properties(0)
    return {'available':True,'name':p.name,'vram_gb':round(p.total_memory/1024**3,1)}

GPU=gpu_info(); print(GPU)
if not GPU.get('available'): raise RuntimeError('Kaggle GPU is not enabled')
router=(job.get('generation') or {}).get('router','auto')
selected='wan' if router=='auto' and GPU.get('vram_gb',0)<24 else ('ltx' if router=='auto' else router)
print('Router selected:',selected)

In [ ]:
# Adapter hook. This cell is intentionally strict: it never publishes.
# Replace/upgrade the model adapter here while keeping the job/result contract unchanged.
def generate_candidate(job, refs, selected, out_dir):
    raise NotImplementedError(
        'Model adapter not installed yet. Wire the validated Wan/LTX Kaggle pipeline here; output must remain candidate-only.'
    )

result={
    'jobId':job_id,
    'state':'candidate-pending-model-adapter',
    'candidateOnly':True,
    'narrativeAuthority':False,
    'router':selected,
    'gpu':GPU,
    'createdAt':time.time(),
    'review':{
        'identityMarion':'pending','identityLucas':'pending','canon':'pending',
        'motion':'pending','continuity':'pending','voice':'pending','wardrobe':'pending','location':'pending'
    }
}
(job_dir/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))

## Output contract
Download the `job_dir` folder as a Kaggle output or upload it back to the repository candidate area using a one-time GitHub/Kaggle connection. **Never write to an approved manifest from this notebook.**